## Flatbug Dataset Metadata: JSON to YAML

This notebook automates the process of merging JSON metadata from the Flatbug dataset into an existing collection of YAML files.

Sources:

Dataset Home: https://darsa.info/flat-bug/

Zenodo Record: https://zenodo.org/records/14761447

## 1. Import Libraries

In [1]:
import yaml
import io
import glob
import os
import json
import datetime

## 2. Locate Source Files
We identify all JSON files in the dataset directory using a recursive search.

In [2]:
# recursive=True must be set, and you must use ** in the pattern
json_files = glob.glob("./photodb_example_2025_12_18/flatbug_dataset/**/*.json", recursive=True)

## 3. Load JSON Metadata
Each JSON file is opened and loaded into a list.

In [3]:
jsonList= []

for json_file in json_files:
    with open(json_file) as json_data:
        z = json.load(json_data)
        jsonList.append(z)     

## 4. Get Existing YAML Files
We crawl the metadata directory to create a list of all existing YAML files that need updates in PhotoDB

In [4]:
# list all yaml files created by PhotoDB
yamlList = []

for root, dirs, files in os.walk('.\\photodb_example_2025_12_18\\flatbug_dataset_meta'):
    for file in files:
        #print(f"  Datei: {os.path.join(root, file)}")
        yamlList.append(os.path.join(root, file))
        # dirs contains names of subdirectories that still need to be chjecked

## 6. Extract Bounding Boxes and Merge Metadata
This code performs the following logic:

- Matches image filenames from JSON to existing YAML paths.
- Merges both.
- Extracts bounding box coordinates for each image ID.
- Appends a timestamped log entry to the record.

In [ ]:
#data=jsonList[1]
#i= data["images"][0]

for data in jsonList:
    for i in data["images"]:
        fileName = next((k for k in yamlList if i["file_name"] in k), None)
        if fileName is None:
            #print(f"Warning: No YAML found for {i['file_name']}")
            continue
        
        # read YAML file
        with open(fileName) as stream:
            data_yaml = yaml.safe_load(stream)
        
        completeDict = data_yaml | i
        # skip already existing files
        # name of output yaml file
        targetFile = 'photodb_example_2025_12_18/flatbug_dataset_full_meta/' + completeDict["location"] + "/" + completeDict["file"] + ".yaml"
        if os.path.exists(targetFile):
           # print(f"Skipping: {targetFile} already exists")
            continue
        
        # 'get id
        target_id = i['id']
        # find the first annotation where image_id matches
        ann = [a for a in data["annotations"] if a["image_id"] == target_id]
        
                # skip if no bounding boxes for the image are available
        if len(ann) <= 0:
            print(f"Warning: No annotation found for image ID {target_id}")
        else:
            w, h = completeDict["width"], completeDict["height"]
            
            # use the "structure below" but add the math
            completeDict['detections'] = [
                {
                    "bbox": [
                        m["bbox"][0] / w,        # X_min
                        m["bbox"][1] / h,        # Y_min
                        (m["bbox"][2] / w), # X_width
                        (m["bbox"][3] / h) # Y_height
                    ],
                    "classifications": [ {"classification": data["categories"][0]["name"], 
                                          "classificator": "manual / semi-automatic", 
                                          "identity": "flatbug",
                                          "date": '2025-01-29T00:00:00'}]
                } 
                for m in ann 
                if m.get("bbox") is not None  # extra safety check
            ]
                
        # create a new dict with licence information, category and info:
        prefixed_license = {f"license_{k}": v for k, v in data["licenses"][0].items()}
        prefixed_info = {f"info_{k}": v for k, v in data["info"].items()}
        #prefixed_categories = {f"categories_{k}": v for k, v in data["categories"][0].items()}
        # Add it to completeDict
        completeDict.update(prefixed_license | prefixed_info)
        
        # add log entry for changed yaml
        completeDict["log"].append({"action": "append JSON metadata", "date": datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S')})
        
        # clean dictionary / remove empty entries
        # identify the keys that have empty string values
        keys_to_remove = [k for k, v in completeDict.items() if v == '']
        
        # delete those keys from the original dictionary
        for k in keys_to_remove:
            del completeDict[k]
        
        # name of new yaml folder
        newpath = 'photodb_example_2025_12_18/flatbug_dataset_full_meta/' + completeDict["location"]
        
        #save to new yaml file 
        if not os.path.exists(newpath):
            os.makedirs(newpath)
        
        with open(targetFile, 'w') as outfile:
            yaml.dump(completeDict, outfile, default_flow_style=False, sort_keys=False) 